In [3]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
from sklearn import metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Load data
filename = 'all.xlsx'
sheetname = 'all'
df = pd.read_excel(filename, sheetname, header=0)

# Drop rows with NaN values
df = df.dropna()

# Define features and target
X = df[['PC_TYPE', 'SS', 'SF%', 'FA%', 'PC', 'w/b', 'b/a', 'SS%', 'FAGG', 'SF', 'FA', 'VOID', 'WR', 'AEA', 'AGE', 'CAGG', 'WATER', 'WR_HR']]
y = df['fc (MPa)']

# Split data into training and testing sets
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the parameter grid
n_estimators = [int(x) for x in np.linspace(start=200, stop=2000, num=10)]
max_features = ['sqrt', 'log2', None]  # 'auto' replaced with None
max_depth = [int(x) for x in np.linspace(10, 110, num=11)]
max_depth.append(None)
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]
bootstrap = [True, False]
random_grid = {
    'n_estimators': n_estimators,
    'max_features': max_features,
    'max_depth': max_depth,
    'min_samples_split': min_samples_split,
    'min_samples_leaf': min_samples_leaf,
    'bootstrap': bootstrap
}

# Initialize RandomForestRegressor and RandomizedSearchCV
rf = RandomForestRegressor()
rf_random = RandomizedSearchCV(estimator=rf, param_distributions=random_grid,
                               n_iter=100, cv=3, verbose=2, random_state=42, n_jobs=-1)
# Fit the model
rf_random.fit(Xtrain, ytrain)

# Get best parameters and estimator
best_params = rf_random.best_params_
rf_model = rf_random.best_estimator_

# Predict and evaluate
random_forest_predict = rf_model.predict(Xtest)
random_forest_R2 = metrics.r2_score(ytest, random_forest_predict)
random_forest_RMSE = metrics.mean_squared_error(ytest, random_forest_predict, squared=False)
random_forest_MAE = metrics.mean_absolute_error(ytest, random_forest_predict)
print(f'R-squared: {random_forest_R2}, RMSE: {random_forest_RMSE}, MAE: {random_forest_MAE}')


[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=2000; total time=   0.0s
[CV] END bootstrap=False, max_depth=50, max_features=auto, min_samples_leaf=1, min_samples_split=2, n_estimators=1000; total time=   0.0s
[CV] END bootstrap=True, max_depth=70, max_features=auto, min_samples_leaf=4, min_samples_split=10, n_estimators=400; total time=   0.0s
[CV] END bootstrap=True, max_depth=None, max_features=auto, min_samples_leaf=2, min_samples_split=2, n_estimators=1800; total time=   0.0s
[CV] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=1000; total time=   0.0s
[CV] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=1000; total time=   0.0s
[CV] END bootstrap=False, max_depth=20, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=1200; total time=   0.0s
[CV] END bootstrap=False, max_depth=

In [4]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
# import matplotlib.pyplot as plt; plt.style.use('seaborn')
import pandas as pd
from sklearn import metrics
from sklearn.model_selection import train_test_split
# from bayes_opt import BayesianOptimization
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

# Load the dataset
filename = 'all.xlsx'
sheetname = 'all'
df = pd.read_excel(filename, sheetname, header=0)
df = df.dropna()

# Define features and target
X = df[['PC_TYPE', 'SS', 'SF%', 'FA%', 'PC', 'w/b', 'b/a', 'SS%', 'FAGG', 'SF', 'FA', 'VOID', 'WR', 'AEA', 'AGE', 'CAGG', 'WATER', 'WR_HR']]
y = df['fc (MPa)']

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_column_name = list(X_train.columns)

# Parameter grid for randomized search
param_grid = {
    'max_depth': np.arange(3, 10, 1),
    'colsample_bytree': np.arange(0.5, 1.0, 0.1),
    'gamma': np.arange(0, 0.5, 0.1),
    'learning_rate': np.arange(0.01, 0.1, 0.01),
    'n_estimators': [100, 200, 300, 400, 500]
}

# Initialize the XGBRegressor
xgb = XGBRegressor(objective='reg:squarederror')

# RandomizedSearchCV for hyperparameter tuning
random_search = RandomizedSearchCV(xgb, param_distributions=param_grid, n_iter=50, scoring='neg_mean_squared_error', cv=3, verbose=3, random_state=42, n_jobs=24)

# Fit the model
random_search.fit(X_train, y_train)

# Get the best model
best_xgb = random_search.best_estimator_

# Make predictions on the test data
predictions = best_xgb.predict(X_test)

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test, predictions)

# Calculate R² score for both train and test sets
train_r2 = r2_score(y_train, best_xgb.predict(X_train))
test_r2 = r2_score(y_test, predictions)

# Print the results
print("Best estimator: ", best_xgb)
print("Best parameters: ", random_search.best_params_)
print("Best validation score: ", random_search.best_score_)
print("MSE on test data: ", mse)
print("R² on training data: ", train_r2)
print("R² on test data: ", test_r2)


Fitting 3 folds for each of 50 candidates, totalling 150 fits
Best estimator:  XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.0, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.09, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)
Best parameters:  {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.09, 'gamma': 0.0, 'colsample_bytree': 0.7}
Best validation score:  -43.204531216180044
MSE on test data: 

In [8]:
import optuna
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error


# 导入数据
df = pd.read_excel('all.xlsx', sheet_name='all')
# 删除缺失值
df.dropna(inplace=True)

X = df[['PC_TYPE', 'SS', 'SF%', 'FA%', 'PC', 'w/b', 'b/a', 'SS%', 'FAGG', 'SF', 'FA', 'VOID', 'WR', 'AEA', 'AGE', 'CAGG', 'WATER', 'WR_HR']].values
Y = df['fc (MPa)'].values
# 提取数据
# Y = df.iloc[:, 0].values
# X = df.iloc[:, 1:].values

# 数据标准化
x_mean = X.mean(0)
x_std = X.std(0)
X_normal = (X - x_mean) / x_std

y_mean = Y.mean()
y_std = Y.std()
Y_normal = (Y - y_mean) / y_std
# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X_normal, Y_normal, train_size=0.80, random_state=42)



def create_model(trial):
    # 为超参数定义搜索空间
    layers = trial.suggest_int('layers', 1, 5)
    neurons = trial.suggest_int('neurons', 16, 256)
    learn_rate = trial.suggest_float('learn_rate', 1e-4, 1e-1,log=True)

    model = Sequential()
    model.add(Dense(neurons, input_dim=X.shape[1], activation='relu', kernel_initializer='he_normal'))
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu', kernel_initializer='he_normal'))
    model.add(Dense(1, activation='linear'))

    optimizer = tf.keras.optimizers.Adam(learn_rate)
    model.compile(loss='mean_squared_error', optimizer=optimizer)
    return model


def objective(trial):
    batch_size = trial.suggest_int('batch_size', 16, 64)

    model = create_model(trial)
    model.fit(X_train, y_train, epochs=100, batch_size=batch_size, verbose=0, validation_split=0.1)

    y_pred = model.predict(X_test)
    return mean_squared_error(y_test, y_pred)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

best_model = create_model(study.best_trial)
batch_size = study.best_trial.params['batch_size']
best_model.fit(X_train, y_train, epochs=300, batch_size=batch_size, verbose=0)

# 保存最优模型
best_model.save('best_model.h5')

# 计算 R2 值
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
print('Train R2:', train_r2)
print('Test R2:', test_r2)

2024-10-25 12:31:27.678779: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[I 2024-10-25 12:31:31,498] A new study created in memory with name: no-name-2a27d7e0-7d92-432c-87c0-2a98ed8a60fe
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


[I 2024-10-25 12:31:47,456] Trial 0 finished with value: 0.6202609252591457 and parameters: {'batch_size': 39, 'layers': 5, 'neurons': 242, 'learn_rate': 0.002567744106437011}. Best is trial 0 with value: 0.6202609252591457.
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


[I 2024-10-25 12:31:56,861] Trial 1 finished with value: 0.5675899263176107 and parameters: {'batch_size': 36, 'layers': 1, 'neurons': 120, 'learn_rate': 0.0136369462193366}. Best is trial 1 with value: 0.5675899263176107.
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


[I 2024-10-25 12:46:26,630] Trial 99 finished with value: 0.5733649189999683 and parameters: {'batch_size': 21, 'layers': 2, 'neurons': 179, 'learn_rate': 0.0032803028154075402}. Best is trial 56 with value: 0.43702608631258894.


Number of finished trials: 100
Best trial: {'batch_size': 23, 'layers': 1, 'neurons': 197, 'learn_rate': 0.003289282795719042}


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 797us/step
Train R2: 0.7451239904738156
Test R2: 0.38902666396078767


## emsemble ANN

In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score

# Custom PyTorch model
class ANNModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(ANNModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Load and preprocess the data
# df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df = pd.read_excel('normal.xlsx', sheet_name='age7')
df.dropna(inplace=True)

X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values
Y = df['fc (MPa)'].values

# Normalize the data
x_mean = X.mean(0)
x_std = X.std(0)
X_normal = (X - x_mean) / x_std

y_mean = Y.mean()
y_std = Y.std()
Y_normal = (Y - y_mean) / y_std

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_normal, Y_normal, test_size=0.2, random_state=42)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Hyperparameters
# params = {'batch_size': 23, 'layers': 3, 'neurons': 197, 'learn_rate': 0.003289282795719042}
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0075788652034274205}

# Define model, loss function, and optimizer
def build_model(input_dim, layers, neurons):
    model = ANNModel(input_dim=input_dim, layers=layers, neurons=neurons)
    return model

# K-Fold Cross-validation setup
kf = KFold(n_splits=100, shuffle=True, random_state=42)
models = []
preds_train = np.zeros_like(y_train)

# Loss function and optimizer
loss_fn = nn.MSELoss()

# Perform K-fold training
for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
    X_train_fold = X_train_tensor[train_index]
    y_train_fold = y_train_tensor[train_index]
    X_val_fold = X_train_tensor[val_index]
    y_val_fold = y_train_tensor[val_index]
    
    model = build_model(input_dim=X_train.shape[1], layers=params['layers'], neurons=params['neurons'])
    optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

    # Training loop
    for epoch in range(300):  # You can increase the number of epochs if necessary
        model.train()
        optimizer.zero_grad()
        y_pred_train = model(X_train_fold)
        loss = loss_fn(y_pred_train, y_train_fold)
        loss.backward()
        optimizer.step()

    # Save the model
    models.append(model)
    
    # Generate validation predictions
    model.eval()
    with torch.no_grad():
        preds_train[val_index] = model(X_val_fold).numpy().flatten()

# Evaluate cross-validation score on the train set
cv_score = r2_score(y_train, preds_train)
print(f'Cross-validation R2 score: {cv_score}')

# Ensemble predictions on test data
preds_test = np.zeros_like(y_test)

for model in models:
    model.eval()
    with torch.no_grad():
        preds_test += model(X_test_tensor).numpy().flatten()

# Average predictions
preds_test /= len(models)

# Calculate R2 score on the test data
test_score = r2_score(y_test, preds_test)
print(f'Test R2 score: {test_score}')

Cross-validation R2 score: 0.2919628201034933
Test R2 score: 0.42903030421662025


In [15]:
import pandas as pd

# Load the new test data
new_test_data = pd.read_excel('test_data.xlsx', sheet_name='age7')

# Ensure the new test data has the same features as the training data
X_new_test = new_test_data[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
                            'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']]

# Predict using the trained model
# fc_predict = best_xgb.predict(X_new_test)

# Ensemble predictions on test data
preds_test = np.zeros_like(y_test)

for model in models:
    model.eval()
    with torch.no_grad():
        preds_test += model(X_test_tensor).numpy().flatten()

# Average predictions
preds_test /= len(models)

# Add predictions as a new column
new_test_data['ANN_emsemble'] = preds_test

# Save the updated test data back into the original file
with pd.ExcelWriter('test_data.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    new_test_data.to_excel(writer, sheet_name='age7', index=False)

print("Predictions saved to 'test_data.xlsx' in the existing sheet 'age7'.")

Predictions saved to 'test_data.xlsx' in the existing sheet 'age7'.


## KINN

In [31]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score

class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Custom loss function using the fitted parameters
def custom_loss(outputs, targets, inputs, a, b):
    mse_loss = nn.MSELoss()(outputs, targets)
    
    # Extract features needed for the fitted equation
    # AGE = inputs[:, 0]  # Assuming AGE is the first feature
    wb = inputs[:, -4]  # Assuming wb is the seventh feature from the end
    
    # Clamp AGE to avoid log of zero or negative numbers
    # AGE = torch.clamp(AGE, min=1e-6)
    
    # Compute the fitted equation: fc = (a * log(AGE) + b) * (e * AGE^d)^(-wb)
    # fc_pred = (a * torch.log(AGE) + b) * (e * torch.pow(AGE, d)) ** (-wb)

    fc_pred = a * b ** (-wb)
    
    # Calculate the residual between ANN predicted outputs and fitted_fc
    residual = torch.abs(outputs - fc_pred.unsqueeze(1))
    
    # Replace any NaNs in the residual with 0.0
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e4, neginf=-1e4)
    
    # Normalize residual by comparing its mean square with the MSE
    mean_square_residual = torch.mean(residual ** 2)
    if mean_square_residual.item() > 0:  # Avoid division by zero
        residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    else:
        residual_normalized = residual  # In case the residual is exactly zero
    
    # Combine MSE loss and the normalized residual
    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    
    return total_loss

# Training function
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24, a=None, b=None):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs, a, b)
            loss.backward()
            optimizer.step()


df = pd.read_excel('normal.xlsx', sheet_name='age7')
df.dropna(inplace=True)

X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values

Y = df['fc (MPa)'].values

# Normalize the data
# x_mean = X.mean(0)
# x_std = X.std(0)
# X_normal = (X - x_mean) / x_std

# y_mean = Y.mean()
# y_std = Y.std()
# Y_normal = (Y - y_mean) / y_std

# Split data into train and test sets
X_train, _, y_train, _ = train_test_split(X, Y, test_size=0.2, random_state=42)

new_test_data = pd.read_excel('test_data.xlsx', sheet_name='age7_remove_outlier')

# Ensure the new test data has the same features as the training data
X_test = new_test_data[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
                            'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values

y_test = new_test_data['fc (MPa)'].values


# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Define the fitted parameters
# a = 40.50
# b = 15.29
a = 151.01590678582858
b = 48.9602723045581
# e = 6.49
# d = 0.36

# Hyperparameters
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0076}

# Define model, loss function, and optimizer
def build_model(input_dim, layers, neurons):
    model = RegressionModel(input_dim=input_dim, layers=layers, neurons=neurons)
    return model

# K-Fold Cross-validation setup
kf = KFold(n_splits=20, shuffle=True, random_state=42)
models = []
preds_train = np.zeros_like(y_train)

# Perform K-fold training
for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
    X_train_fold = X_train_tensor[train_index]
    y_train_fold = y_train_tensor[train_index]
    X_val_fold = X_train_tensor[val_index]
    y_val_fold = y_train_tensor[val_index]
    
    model = build_model(input_dim=X_train.shape[1], layers=params['layers'], neurons=params['neurons'])
    optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

    # Train the model
    train_model(model, optimizer, X_train_fold, y_train_fold, epochs=300, batch_size=params['batch_size'], a=a, b=b)

    # Save the model
    models.append(model)
    
    # Generate validation predictions
    model.eval()
    with torch.no_grad():
        preds_train[val_index] = model(X_val_fold).numpy().flatten()

# Evaluate cross-validation score on the train set
cv_score = r2_score(y_train, preds_train)
print(f'Cross-validation R2 score: {cv_score}')

# Ensemble predictions on test data
preds_test = np.zeros_like(y_test)

for model in models:
    model.eval()
    with torch.no_grad():
        preds_test += model(X_test_tensor).numpy().flatten()

# Average predictions
preds_test /= len(models)

# Calculate R2 score on the test data
test_score = r2_score(y_test, preds_test)
print(f'Test R2 score: {test_score}')


Cross-validation R2 score: -3.1200847214478724
Test R2 score: 0.5695162980304527


In [32]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Custom loss function
def custom_loss(outputs, targets, inputs, a, b):
    mse_loss = nn.MSELoss()(outputs, targets)
    wb = inputs[:, -4]
    fc_pred = a * b ** (-wb)
    residual = torch.abs(outputs - fc_pred.unsqueeze(1))
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e4, neginf=-1e4)
    mean_square_residual = torch.mean(residual ** 2)
    if mean_square_residual.item() > 0:
        residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    else:
        residual_normalized = residual
    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    return total_loss

# Training function
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24, a=None, b=None):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs, a, b)
            loss.backward()
            optimizer.step()

# Load and split data
df = pd.read_excel('normal.xlsx', sheet_name='age7')
df.dropna(inplace=True)
X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values
Y = df['fc (MPa)'].values

# Split data into 80% training and 20% validation set
X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

# Standardize the training and validation data
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
X_val_scaled = scaler_X.transform(X_val)
y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1)).flatten()

# Load new test data
new_test_data = pd.read_excel('test_data.xlsx', sheet_name='age7_remove_outlier')
X_test = new_test_data[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
                        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values
y_test = new_test_data['fc (MPa)'].values
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32).view(-1, 1)

# Fitted parameters
a = 151.01590678582858
b = 48.9602723045581

# Hyperparameters
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0076}

# Define model, loss function, and optimizer
def build_model(input_dim, layers, neurons):
    model = RegressionModel(input_dim=input_dim, layers=layers, neurons=neurons)
    return model

# K-Fold Cross-validation setup
kf = KFold(n_splits=20, shuffle=True, random_state=42)
models = []
preds_train = np.zeros_like(y_train_scaled)

# Perform K-fold training
for fold, (train_index, val_index) in enumerate(kf.split(X_train_scaled)):
    X_train_fold = X_train_tensor[train_index]
    y_train_fold = y_train_tensor[train_index]
    X_val_fold = X_train_tensor[val_index]
    y_val_fold = y_train_tensor[val_index]
    
    model = build_model(input_dim=X_train_scaled.shape[1], layers=params['layers'], neurons=params['neurons'])
    optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

    # Train the model
    train_model(model, optimizer, X_train_fold, y_train_fold, epochs=300, batch_size=params['batch_size'], a=a, b=b)

    # Save the model
    models.append(model)
    
    # Generate validation predictions
    model.eval()
    with torch.no_grad():
        preds_train[val_index] = model(X_val_fold).numpy().flatten()

# Evaluate cross-validation score on the train set
# cv_score = r2_score(y_train_scaled, preds_train)
# print(f'Cross-validation R2 score: {cv_score}')

# Ensemble predictions on test data
preds_test = np.zeros_like(y_test_scaled)
for model in models:
    model.eval()
    with torch.no_grad():
        preds_test += model(X_test_tensor).numpy().flatten()

# Average predictions and rescale
preds_test /= len(models)
preds_test_rescaled = scaler_y.inverse_transform(preds_test.reshape(-1, 1)).flatten()

# Calculate R2 score on the test data
test_score = r2_score(y_test, preds_test_rescaled)
print(f'Test R2 score: {test_score}')


Cross-validation R2 score: -0.8440640532411419
Test R2 score: 0.6031232287471003


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Custom loss function
def custom_loss(outputs, targets, inputs, a, b):
    mse_loss = nn.MSELoss()(outputs, targets)
    wb = inputs[:, -4]
    fc_pred = a * b ** (-wb)



    
    residual = torch.abs(outputs - fc_pred.unsqueeze(1))
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e4, neginf=-1e4)
    mean_square_residual = torch.mean(residual ** 2)
    if mean_square_residual.item() > 0:
        residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    else:
        residual_normalized = residual
    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    return total_loss

# Training function
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24, a=None, b=None):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs, a, b)
            loss.backward()
            optimizer.step()

# Load and split data
df = pd.read_excel('normal.xlsx', sheet_name='age7')
df.dropna(inplace=True)
X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values
Y = df['fc (MPa)'].values

# Split data into 80% training and 20% validation set
X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

# Standardize the training and validation data
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
X_val_scaled = scaler_X.transform(X_val)
y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1)).flatten()

# Load new test data
new_test_data = pd.read_excel('test_data.xlsx', sheet_name='age7_remove_outlier')
X_test = new_test_data[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
                        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values
y_test = new_test_data['fc (MPa)'].values
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32).view(-1, 1)

# Fitted parameters
a = 151.01590678582858
b = 48.9602723045581

# Hyperparameters
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0076}

# Define model, loss function, and optimizer
def build_model(input_dim, layers, neurons):
    model = RegressionModel(input_dim=input_dim, layers=layers, neurons=neurons)
    return model

# K-Fold Cross-validation setup
kf = KFold(n_splits=20, shuffle=True, random_state=42)
models = []
preds_train = np.zeros_like(y_train_scaled)

# Perform K-fold training
for fold, (train_index, val_index) in enumerate(kf.split(X_train_scaled)):
    X_train_fold = X_train_tensor[train_index]
    y_train_fold = y_train_tensor[train_index]
    X_val_fold = X_train_tensor[val_index]
    y_val_fold = y_train_tensor[val_index]
    
    model = build_model(input_dim=X_train_scaled.shape[1], layers=params['layers'], neurons=params['neurons'])
    optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

    # Train the model
    train_model(model, optimizer, X_train_fold, y_train_fold, epochs=300, batch_size=params['batch_size'], a=a, b=b)

    # Save the model
    models.append(model)
    
    # Generate validation predictions
    model.eval()
    with torch.no_grad():
        preds_train[val_index] = model(X_val_fold).numpy().flatten()

# Evaluate cross-validation score on the train set
# cv_score = r2_score(y_train_scaled, preds_train)
# print(f'Cross-validation R2 score: {cv_score}')

# Ensemble predictions on test data
preds_test = np.zeros_like(y_test_scaled)
for model in models:
    model.eval()
    with torch.no_grad():
        preds_test += model(X_test_tensor).numpy().flatten()

# Average predictions and rescale
preds_test /= len(models)
preds_test_rescaled = scaler_y.inverse_transform(preds_test.reshape(-1, 1)).flatten()

# Calculate R2 score on the test data
test_score = r2_score(y_test, preds_test_rescaled)
print(f'Test R2 score: {test_score}')
